In [2]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/SimoRinaldi/crop-spatial-classification.git
    else:
        !cd {REPO_DIR} && git pull
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    !pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato!")
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

Ambiente Colab rilevato. Inizializzazione in corso...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.22 KiB | 2.22 MiB/s, done.
From https://github.com/SimoRinaldi/crop-spatial-classification
   548ac97..e93af33  main       -> origin/main
Updating 548ac97..e93af33
Fast-forward
 notebooks/04_fit_random_forest.ipynb | 142 +++++++++++++++++++++++++++++++++++
 1 file changed, 142 insertions(+)
 create mode 100644 notebooks/04_fit_random_forest.ipynb
Setup ambiente Colab completato!
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/MyDrive/Progetto_MLDM/data/raw


In [3]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

dataset_parquet = Path(f"{DATA_DIR}/processed/dataset_ml.parquet")

dataset = pd.read_parquet(dataset_parquet)

y = dataset['Ground_Truth'].astype(int)

# 8 Feature (Bande + Indici)
X = dataset[['Blu_B02', 'Verde_B03', 'Rosso_B04', 'NIR_B08', 'SWIR1_B11', 'SWIR2_B12', 'NDVI', 'NDWI']]

try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    modello = RandomForestClassifier(n_estimators=100, random_state=42)
    modello.fit(X_train, y_train)
    predizioni = modello.predict(X_test)

    print("🏆 Classification Report 🏆")
    print("-" * 50)
    print(classification_report(y_test, predizioni, zero_division=0))
except Exception as e:
    print(f"Non ho abbastanza dati per fare il test. Errore: {e}")

🏆 Classification Report 🏆
--------------------------------------------------
              precision    recall  f1-score   support

        1110       0.42      0.50      0.46        32
        1120       0.22      0.33      0.27        12
        1130       0.00      0.00      0.00         2
        1150       0.00      0.00      0.00         0
        1210       0.00      0.00      0.00         7
        1220       0.33      0.14      0.20         7
        1410       0.00      0.00      0.00         1
        2100       0.20      0.29      0.24         7
        2200       0.59      0.61      0.60        44
        2310       0.33      0.14      0.20         7
        3100       0.00      0.00      0.00         1

    accuracy                           0.42       120
   macro avg       0.19      0.18      0.18       120
weighted avg       0.40      0.42      0.41       120

